# 🧮 Module 1.4: Image as Matrix — Linear Algebra of Vision

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_01_Image_Foundations/04_Image_As_Matrix/image_as_matrix.ipynb)

---

## 🎯 Learning Objectives
1. Image operations as matrix operations
2. SVD decomposition of images
3. Eigenfaces and dimensionality reduction
4. How RL uses compressed image representations

---

In [ ]:
!pip install numpy matplotlib opencv-python-headless scikit-image -q

import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import svd

print("✅ Ready!")

In [ ]:
# Download REAL open-source dataset
from skimage import data
import torchvision

# Real images from scikit-image (built-in, no download needed)
astronaut_img = data.astronaut()       # 512x512x3 real photo
camera_img = data.camera()             # 512x512 grayscale real photo  
coins_img = data.coins()               # Real coins photograph
coffee_img = data.coffee()             # 400x600x3 real photo

# MNIST - 70,000 real handwritten digits (auto-downloads ~11MB)
mnist_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True)
print(f"✅ MNIST loaded: {len(mnist_dataset)} real handwritten digit images (28x28)")
print(f"✅ scikit-image loaded: astronaut {astronaut_img.shape}, camera {camera_img.shape}")

## Deep Mathematical Derivation: SVD and Image Compression

### Step 1: Singular Value Decomposition (Existence Proof)
**Theorem:** Every matrix $\mathbf{A} \in \mathbb{R}^{m \times n}$ has a decomposition:
$$\mathbf{A} = \mathbf{U} \boldsymbol{\Sigma} \mathbf{V}^T$$

where $\mathbf{U} \in \mathbb{R}^{m \times m}$ (orthogonal), $\boldsymbol{\Sigma} \in \mathbb{R}^{m \times n}$ (diagonal, non-negative), $\mathbf{V} \in \mathbb{R}^{n \times n}$ (orthogonal).

**Proof sketch:**
1. $\mathbf{A}^T\mathbf{A}$ is symmetric positive semi-definite, so it has eigendecomposition: $\mathbf{A}^T\mathbf{A} = \mathbf{V}\boldsymbol{\Lambda}\mathbf{V}^T$
2. Define singular values: $\sigma_i = \sqrt{\lambda_i}$ where $\lambda_i$ are eigenvalues of $\mathbf{A}^T\mathbf{A}$
3. For each $\sigma_i > 0$, define: $\mathbf{u}_i = \frac{1}{\sigma_i}\mathbf{A}\mathbf{v}_i$
4. Verify orthonormality: $\mathbf{u}_i^T\mathbf{u}_j = \frac{1}{\sigma_i\sigma_j}\mathbf{v}_i^T\mathbf{A}^T\mathbf{A}\mathbf{v}_j = \frac{\lambda_j}{\sigma_i\sigma_j}\mathbf{v}_i^T\mathbf{v}_j = \delta_{ij}$ $\blacksquare$

### Step 2: Low-Rank Approximation (Eckart-Young-Mirsky Theorem)
**Theorem:** The best rank-$k$ approximation to $\mathbf{A}$ (in Frobenius norm) is:
$$\mathbf{A}_k = \sum_{i=1}^k \sigma_i \mathbf{u}_i \mathbf{v}_i^T$$

**Proof:**
$$\|\mathbf{A} - \mathbf{A}_k\|_F^2 = \sum_{i=k+1}^r \sigma_i^2$$

For any other rank-$k$ matrix $\mathbf{B}$:
$$\|\mathbf{A} - \mathbf{B}\|_F^2 \geq \sum_{i=k+1}^r \sigma_i^2 = \|\mathbf{A} - \mathbf{A}_k\|_F^2$$

This follows from the Courant-Fischer minimax theorem applied to singular values. $\blacksquare$

### Step 3: Compression Ratio Analysis
**Original storage:** $m \times n$ values

**Rank-$k$ SVD storage:** $k(m + n + 1)$ values (k left singular vectors + k right singular vectors + k singular values)

**Compression ratio:**
$$\rho = \frac{mn}{k(m + n + 1)} \approx \frac{mn}{k(m+n)}$$

**For a 512×512 grayscale image with $k=50$:**
$$\rho = \frac{512 \times 512}{50 \times (512 + 512 + 1)} \approx 5.1\times \text{ compression}$$

### Step 4: Eigenfaces (PCA on Image Space)
Given $N$ face images $\mathbf{x}_1, \ldots, \mathbf{x}_N$, each flattened to $\mathbb{R}^d$:

**Step 4a:** Compute mean face: $\boldsymbol{\mu} = \frac{1}{N}\sum_{i=1}^N \mathbf{x}_i$

**Step 4b:** Center the data: $\boldsymbol{\Phi}_i = \mathbf{x}_i - \boldsymbol{\mu}$

**Step 4c:** Covariance matrix: $\mathbf{C} = \frac{1}{N}\sum_{i=1}^N \boldsymbol{\Phi}_i \boldsymbol{\Phi}_i^T = \frac{1}{N}\mathbf{X}\mathbf{X}^T$

**Step 4d:** Since $\mathbf{C} \in \mathbb{R}^{d \times d}$ is huge, use the trick: compute eigenvectors of $\mathbf{X}^T\mathbf{X} \in \mathbb{R}^{N \times N}$ instead.

**Proof of equivalence:** If $\mathbf{X}^T\mathbf{X}\mathbf{v} = \lambda\mathbf{v}$, then $\mathbf{X}\mathbf{X}^T(\mathbf{X}\mathbf{v}) = \lambda(\mathbf{X}\mathbf{v})$, so $\mathbf{u} = \mathbf{X}\mathbf{v}/\|\mathbf{X}\mathbf{v}\|$ is an eigenvector of $\mathbf{C}$. $\blacksquare$

### Step 5: RL Connection — Dimensionality Reduction for State Space
Full image state space: $|S| = 256^{H \times W}$ (intractable)

With rank-$k$ SVD: $|S_{\text{compressed}}| = 256^{k(H+W+1)/HW \cdot HW}$ — exponentially smaller!

**This is why autoencoders (learned nonlinear SVD) are used in RL to compress image states into manageable latent representations.**

## Eckart-Young-Mirsky Theorem — Optimal Low-Rank Approximation (Full Proof)

The Eckart-Young theorem is the mathematical backbone of SVD-based image compression. It guarantees that truncated SVD gives the **best possible** low-rank approximation.

### Theorem Statement

**Eckart-Young-Mirsky (1936):** Let $\mathbf{A} \in \mathbb{R}^{m \times n}$ with SVD $\mathbf{A} = \sum_{i=1}^{r} \sigma_i \mathbf{u}_i \mathbf{v}_i^T$ where $\sigma_1 \geq \sigma_2 \geq \cdots \geq \sigma_r > 0$. Then for any matrix $\mathbf{B}$ with $\operatorname{rank}(\mathbf{B}) \leq k$:

$$\|\mathbf{A} - \mathbf{A}_k\|_F \leq \|\mathbf{A} - \mathbf{B}\|_F$$

where $\mathbf{A}_k = \sum_{i=1}^{k} \sigma_i \mathbf{u}_i \mathbf{v}_i^T$ is the rank-$k$ truncated SVD.

### Step 1: Express the Frobenius Norm Error

The Frobenius norm of a matrix is $\|\mathbf{M}\|_F = \sqrt{\sum_{i,j} M_{ij}^2} = \sqrt{\operatorname{tr}(\mathbf{M}^T\mathbf{M})}$.

For the truncated SVD:
$$\mathbf{A} - \mathbf{A}_k = \sum_{i=k+1}^{r} \sigma_i \mathbf{u}_i \mathbf{v}_i^T$$

Since $\{\mathbf{u}_i\}$ and $\{\mathbf{v}_i\}$ are orthonormal:
$$\|\mathbf{A} - \mathbf{A}_k\|_F^2 = \sum_{i=k+1}^{r} \sigma_i^2$$

### Step 2: Prove Optimality via the Courant-Fischer Minimax Theorem

**Courant-Fischer Theorem:** The $i$-th largest singular value of $\mathbf{A}$ satisfies:
$$\sigma_i = \min_{\dim(V) = n - i + 1} \max_{\mathbf{x} \in V, \|\mathbf{x}\|=1} \|\mathbf{A}\mathbf{x}\|_2$$

Let $\mathbf{B}$ be any rank-$k$ matrix. Then $\ker(\mathbf{B})$ has dimension $\geq n - k$.

Consider the subspace $W = \operatorname{span}\{\mathbf{v}_1, \ldots, \mathbf{v}_{k+1}\}$ with $\dim(W) = k + 1$.

By dimension counting: $\dim(W) + \dim(\ker(\mathbf{B})) = (k+1) + (n-k) = n + 1 > n$.

Therefore $W \cap \ker(\mathbf{B}) \neq \{0\}$. Pick unit $\mathbf{z} \in W \cap \ker(\mathbf{B})$.

### Step 3: Lower Bound on Approximation Error

Write $\mathbf{z} = \sum_{i=1}^{k+1} \alpha_i \mathbf{v}_i$ with $\sum_{i=1}^{k+1} \alpha_i^2 = 1$.

$$\|\mathbf{A} - \mathbf{B}\|_F^2 \geq \|(\mathbf{A} - \mathbf{B})\mathbf{z}\|_2^2 = \|\mathbf{A}\mathbf{z}\|_2^2 \quad (\text{since } \mathbf{B}\mathbf{z} = 0)$$

$$= \left\|\sum_{i=1}^{k+1} \alpha_i \sigma_i \mathbf{u}_i\right\|_2^2 = \sum_{i=1}^{k+1} \alpha_i^2 \sigma_i^2 \geq \sigma_{k+1}^2 \sum_{i=1}^{k+1} \alpha_i^2 = \sigma_{k+1}^2$$

Extending this argument for all singular values $\sigma_{k+1}, \ldots, \sigma_r$:

$$\|\mathbf{A} - \mathbf{B}\|_F^2 \geq \sum_{i=k+1}^{r} \sigma_i^2 = \|\mathbf{A} - \mathbf{A}_k\|_F^2 \quad \blacksquare$$

### Image Compression Interpretation

For a 512×512 grayscale image with rank $r = 512$ and rapidly decaying singular values:
- Using $k = 10$ components: error $= \sqrt{\sum_{i=11}^{512} \sigma_i^2}$, storage $= 10 \times (512 + 512 + 1) = 10{,}250$ values vs $262{,}144$ original
- The theorem guarantees **no other method** using only 10 basis images can do better
- This is the mathematical reason why SVD compression is optimal among linear methods

## PCA, Eigendecomposition, and the Covariance Connection

Principal Component Analysis on images is intimately connected to SVD through the eigendecomposition of covariance matrices. This connection is the foundation of Eigenfaces.

### Step 1: Eigendecomposition of $\mathbf{A}^T\mathbf{A}$ and $\mathbf{A}\mathbf{A}^T$

Given $\mathbf{A} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^T$:

$$\mathbf{A}^T\mathbf{A} = (\mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^T)^T(\mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^T) = \mathbf{V}\boldsymbol{\Sigma}^T\mathbf{U}^T\mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^T = \mathbf{V}\boldsymbol{\Sigma}^2\mathbf{V}^T$$

$$\mathbf{A}\mathbf{A}^T = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^T\mathbf{V}\boldsymbol{\Sigma}^T\mathbf{U}^T = \mathbf{U}\boldsymbol{\Sigma}^2\mathbf{U}^T$$

**Key insight:** The right singular vectors $\mathbf{v}_i$ are eigenvectors of $\mathbf{A}^T\mathbf{A}$, and the left singular vectors $\mathbf{u}_i$ are eigenvectors of $\mathbf{A}\mathbf{A}^T$. The eigenvalues of both are $\sigma_i^2$.

### Step 2: PCA as SVD of Centered Data

Given $N$ images $\{\mathbf{x}_1, \ldots, \mathbf{x}_N\} \subset \mathbb{R}^d$ (each image flattened to a vector), form the centered data matrix:

$$\mathbf{X} = [\mathbf{x}_1 - \bar{\mathbf{x}}, \; \mathbf{x}_2 - \bar{\mathbf{x}}, \; \ldots, \; \mathbf{x}_N - \bar{\mathbf{x}}] \in \mathbb{R}^{d \times N}$$

The sample covariance matrix is $\mathbf{C} = \frac{1}{N-1}\mathbf{X}\mathbf{X}^T \in \mathbb{R}^{d \times d}$.

PCA finds the eigenvectors of $\mathbf{C}$, which are exactly the left singular vectors of $\mathbf{X}$:

$$\mathbf{X} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^T \implies \mathbf{C} = \frac{1}{N-1}\mathbf{U}\boldsymbol{\Sigma}^2\mathbf{U}^T$$

### Step 3: The Kernel Trick for High-Dimensional Images

For images, $d = H \times W$ can be very large (e.g., $d = 262{,}144$ for 512×512), making $\mathbf{C} \in \mathbb{R}^{d \times d}$ intractable.

**The trick:** Instead of eigendecomposing $\mathbf{X}\mathbf{X}^T \in \mathbb{R}^{d \times d}$, decompose $\mathbf{X}^T\mathbf{X} \in \mathbb{R}^{N \times N}$:

$$\mathbf{X}^T\mathbf{X} \mathbf{v}_i = \sigma_i^2 \mathbf{v}_i \implies \mathbf{u}_i = \frac{1}{\sigma_i}\mathbf{X}\mathbf{v}_i$$

**Proof of correctness:**
$$\mathbf{X}\mathbf{X}^T \mathbf{u}_i = \mathbf{X}\mathbf{X}^T \cdot \frac{1}{\sigma_i}\mathbf{X}\mathbf{v}_i = \frac{1}{\sigma_i}\mathbf{X}(\mathbf{X}^T\mathbf{X})\mathbf{v}_i = \frac{1}{\sigma_i}\mathbf{X} \cdot \sigma_i^2 \mathbf{v}_i = \sigma_i^2 \mathbf{u}_i \quad \blacksquare$$

**Complexity reduction:** From $O(d^3)$ to $O(N^3)$. For Eigenfaces with $N = 1000$ face images of size $d = 65{,}536$ (256×256), this is a reduction from $O(10^{14})$ to $O(10^9)$ — a factor of $10^5$!

### Step 4: Projection and Reconstruction

Any image $\mathbf{x}$ can be projected onto the $k$-dimensional PCA subspace:
$$\mathbf{z} = \mathbf{U}_k^T (\mathbf{x} - \bar{\mathbf{x}}) \in \mathbb{R}^k$$

Reconstruction:
$$\hat{\mathbf{x}} = \bar{\mathbf{x}} + \mathbf{U}_k \mathbf{z}$$

The reconstruction error is bounded by the discarded eigenvalues:
$$\|\mathbf{x} - \hat{\mathbf{x}}\|^2 \leq \sum_{i=k+1}^{d} \lambda_i$$

### RL Application: Compressed State Representation

In RL, the PCA projection $\mathbf{z} = \mathbf{U}_k^T(\mathbf{x} - \bar{\mathbf{x}})$ serves as a compressed state:
- Full state: $\mathbf{x} \in \mathbb{R}^{65536}$ → intractable for tabular RL
- PCA state: $\mathbf{z} \in \mathbb{R}^{50}$ → manageable, captures 95%+ variance
- This is the linear precursor to the nonlinear compression learned by autoencoders in deep RL

## Image Rank and Intrinsic Dimensionality

### Rank of an Image Matrix

The rank of an image matrix $\mathbf{I} \in \mathbb{R}^{m \times n}$ is the number of nonzero singular values:

$$\text{rank}(\mathbf{I}) = |\{i : \sigma_i > 0\}|$$

**Maximum rank:** $\min(m, n)$. A generic $m \times n$ image has full rank, but the **effective rank** (number of significant singular values) is typically much smaller.

### Effective Rank (Roy-Vetterli Definition)

$$\text{erank}(\mathbf{I}) = \exp\left(-\sum_{i=1}^{r} \hat{\sigma}_i \log \hat{\sigma}_i\right)$$

where $\hat{\sigma}_i = \sigma_i / \sum_j \sigma_j$ are the normalized singular values.

**Proof that $1 \leq \text{erank} \leq r$:**
- Minimum: when only one $\sigma$ is nonzero, $\hat{\sigma}_1 = 1$, entropy $= 0$, erank $= e^0 = 1$
- Maximum: when all $\sigma_i$ are equal, $\hat{\sigma}_i = 1/r$, entropy $= \log r$, erank $= e^{\log r} = r$ $\blacksquare$

### Numerical Rank (Practical Definition)

$$\text{nrank}_\epsilon(\mathbf{I}) = |\{i : \sigma_i > \epsilon \cdot \sigma_1\}|$$

For $\epsilon = 0.01$ (1% threshold), this counts singular values above 1% of the largest. Typical values for natural 512×512 images: nrank $\approx 50-150$, far below the maximum of 512.

### Implications for Representation Learning

The low effective rank of images is the mathematical basis for:
1. **SVD compression:** Keep only $k \ll \min(m,n)$ components
2. **Autoencoders:** The bottleneck dimension should match the effective rank
3. **RL state compression:** The compressed state dimension should be $\geq$ effective rank to avoid information loss

**Rule of thumb:** Set the compressed dimension to capture 95% of the singular value energy:
$$k_{95} = \min\left\{k : \frac{\sum_{i=1}^k \sigma_i^2}{\sum_{i=1}^r \sigma_i^2} \geq 0.95\right\}$$

## 1. Image = Matrix

A grayscale image of size $M \times N$ is a matrix:
$$\mathbf{I} \in \mathbb{R}^{M \times N}$$

### Matrix Operations on Images:

**Transpose:** $\mathbf{I}^T$ flips the image along the diagonal

**Matrix multiplication:** $\mathbf{I}_{out} = \mathbf{A} \cdot \mathbf{I}$ applies a linear transformation

**Element-wise:** $\mathbf{I}_{out} = \mathbf{I}_1 \odot \mathbf{I}_2$ (Hadamard product) for masking

## 2. SVD — Compressing Images

Any image matrix can be decomposed:
$$\mathbf{I} = \mathbf{U} \boldsymbol{\Sigma} \mathbf{V}^T = \sum_{i=1}^{r} \sigma_i \mathbf{u}_i \mathbf{v}_i^T$$

Where:
- $\mathbf{U} \in \mathbb{R}^{M \times M}$ — left singular vectors
- $\boldsymbol{\Sigma} \in \mathbb{R}^{M \times N}$ — singular values (diagonal)
- $\mathbf{V}^T \in \mathbb{R}^{N \times N}$ — right singular vectors

### Low-rank approximation (compression):
$$\mathbf{I}_k = \sum_{i=1}^{k} \sigma_i \mathbf{u}_i \mathbf{v}_i^T \quad (k < r)$$

**Compression ratio:** $\frac{k(M + N + 1)}{MN}$

### Why This Matters for RL:
RL agents need **compact state representations**. SVD shows that images have low intrinsic dimensionality — we can represent them with far fewer numbers!

## Matrix Norms — Frobenius, Spectral, and Nuclear Norms

Different matrix norms measure different aspects of an image matrix. Understanding their relationships is critical for image quality assessment and compression theory.

### The Frobenius Norm

$$\|\mathbf{A}\|_F = \sqrt{\sum_{i=1}^m \sum_{j=1}^n |a_{ij}|^2} = \sqrt{\operatorname{tr}(\mathbf{A}^T\mathbf{A})}$$

**Connection to SVD:**
$$\|\mathbf{A}\|_F = \sqrt{\sum_{i=1}^r \sigma_i^2}$$

**Proof:**
$$\|\mathbf{A}\|_F^2 = \operatorname{tr}(\mathbf{A}^T\mathbf{A}) = \operatorname{tr}(\mathbf{V}\boldsymbol{\Sigma}^2\mathbf{V}^T) = \operatorname{tr}(\boldsymbol{\Sigma}^2\mathbf{V}^T\mathbf{V}) = \operatorname{tr}(\boldsymbol{\Sigma}^2) = \sum_{i=1}^r \sigma_i^2$$

using the cyclic property of the trace: $\operatorname{tr}(\mathbf{ABC}) = \operatorname{tr}(\mathbf{CAB})$. $\blacksquare$

**Image interpretation:** $\|\mathbf{A}\|_F^2$ equals the total "energy" (sum of squared pixel values) of the image.

### The Spectral Norm (Operator Norm)

$$\|\mathbf{A}\|_2 = \max_{\|\mathbf{x}\|_2 = 1} \|\mathbf{A}\mathbf{x}\|_2 = \sigma_1$$

**Proof:** Since $\mathbf{A}\mathbf{v}_i = \sigma_i\mathbf{u}_i$ for any unit vector $\mathbf{x} = \sum_i \alpha_i\mathbf{v}_i$ with $\sum \alpha_i^2 = 1$:
$$\|\mathbf{A}\mathbf{x}\|^2 = \left\|\sum_i \alpha_i\sigma_i\mathbf{u}_i\right\|^2 = \sum_i \alpha_i^2\sigma_i^2 \leq \sigma_1^2\sum_i\alpha_i^2 = \sigma_1^2$$

Equality when $\mathbf{x} = \mathbf{v}_1$. $\blacksquare$

**Image interpretation:** $\sigma_1$ captures the dominant spatial pattern in the image — the single "most important" feature.

### The Nuclear Norm (Trace Norm)

$$\|\mathbf{A}\|_* = \sum_{i=1}^r \sigma_i$$

**The nuclear norm is the tightest convex relaxation of the rank function.** This makes it crucial for low-rank matrix completion and image inpainting:

$$\min_{\mathbf{X}} \|\mathbf{X}\|_* \quad \text{subject to} \quad X_{ij} = A_{ij} \text{ for observed pixels}$$

### Norm Inequalities

**Relationship between norms:**
$$\|\mathbf{A}\|_2 \leq \|\mathbf{A}\|_F \leq \sqrt{r}\|\mathbf{A}\|_2$$

**Proof of upper bound:** $\|\mathbf{A}\|_F = \sqrt{\sum_{i=1}^r \sigma_i^2} \leq \sqrt{r \cdot \sigma_1^2} = \sqrt{r}\|\mathbf{A}\|_2$ $\blacksquare$

**Proof of lower bound:** $\|\mathbf{A}\|_F = \sqrt{\sum \sigma_i^2} \geq \sqrt{\sigma_1^2} = \sigma_1 = \|\mathbf{A}\|_2$ $\blacksquare$

$$\|\mathbf{A}\|_F \leq \|\mathbf{A}\|_* \leq \sqrt{r}\|\mathbf{A}\|_F$$

### Quality Metrics as Norm Functions

Image quality metrics are norm-based comparisons:
- **MSE** $= \frac{1}{mn}\|\mathbf{A} - \mathbf{B}\|_F^2$
- **PSNR** $= 10\log_{10}\frac{255^2 \cdot mn}{\|\mathbf{A} - \mathbf{B}\|_F^2}$

These metrics inherit the mathematical properties of the Frobenius norm, including triangle inequality and positive definiteness — guaranteeing they behave as proper distance measures.

## Matrix Factorization for Image Inpainting — Completing Missing Data

When parts of an image are missing (corruption, occlusion), matrix completion recovers the missing entries using the low-rank structure.

### Step 1: The Image Inpainting Problem

Given an image matrix $\mathbf{A}$ with observed entries at positions $\Omega \subset \{1,\ldots,m\} \times \{1,\ldots,n\}$:

$$\min_{\mathbf{X}} \operatorname{rank}(\mathbf{X}) \quad \text{s.t. } X_{ij} = A_{ij} \text{ for all } (i,j) \in \Omega$$

Find the lowest-rank matrix consistent with the observations.

### Step 2: Nuclear Norm Relaxation

The rank minimization is NP-hard. The convex relaxation replaces rank with the nuclear norm:

$$\min_{\mathbf{X}} \|\mathbf{X}\|_* \quad \text{s.t. } X_{ij} = A_{ij} \text{ for all } (i,j) \in \Omega$$

**Theorem (Candès & Recht, 2009):** If the rank-$r$ matrix $\mathbf{A}$ satisfies certain incoherence conditions, and $|\Omega| \geq C \cdot rn\log^2 n$ entries are observed uniformly at random, then nuclear norm minimization recovers $\mathbf{A}$ exactly with high probability.

### Step 3: Alternating Least Squares (ALS) Algorithm

Factorize $\mathbf{X} = \mathbf{U}\mathbf{V}^T$ where $\mathbf{U} \in \mathbb{R}^{m \times k}$, $\mathbf{V} \in \mathbb{R}^{n \times k}$:

$$\min_{\mathbf{U}, \mathbf{V}} \sum_{(i,j) \in \Omega} (A_{ij} - \mathbf{u}_i^T\mathbf{v}_j)^2 + \lambda(\|\mathbf{U}\|_F^2 + \|\mathbf{V}\|_F^2)$$

**Alternating steps:**
1. Fix $\mathbf{V}$, solve for each row $\mathbf{u}_i$ (least squares)
2. Fix $\mathbf{U}$, solve for each row $\mathbf{v}_j$ (least squares)

Each step is a convex problem with closed-form solution. The algorithm converges to a local minimum (global minimum for convex relaxations).

### Step 4: Connection to Collaborative Filtering

This is the same mathematical framework as Netflix's recommendation system: users $\leftrightarrow$ image rows, movies $\leftrightarrow$ image columns, ratings $\leftrightarrow$ pixel values. The "taste vectors" $\mathbf{u}_i, \mathbf{v}_j$ are analogous to spatial basis functions.

### Step 5: RL for Adaptive Inpainting

An RL agent can decide inpainting strategy:
- **State:** observed pixels + mask of missing regions + current estimate
- **Action:** adjust rank $k$, regularization $\lambda$, or select inpainting algorithm
- **Reward:** PSNR/SSIM of inpainted result vs. ground truth

The agent learns that textured regions need higher rank, while smooth regions can be completed with $k = 2-3$ components.

## SVD Uniqueness and the Singular Value Spectrum of Natural Images

### Uniqueness of the SVD

**Theorem:** The singular values $\sigma_1 \geq \sigma_2 \geq \cdots \geq \sigma_r > 0$ of $\mathbf{A}$ are unique. The singular vectors are unique when all singular values are distinct.

**Proof of singular value uniqueness:** The singular values are $\sigma_i = \sqrt{\lambda_i(\mathbf{A}^T\mathbf{A})}$. Since $\mathbf{A}^T\mathbf{A}$ is symmetric PSD, its eigenvalues are uniquely determined by the characteristic polynomial $\det(\mathbf{A}^T\mathbf{A} - \lambda\mathbf{I}) = 0$. $\blacksquare$

**Non-uniqueness of singular vectors:** When $\sigma_i = \sigma_j$ (repeated singular values), any rotation within the corresponding eigenspace gives valid singular vectors. For images, repeated singular values are rare except for highly symmetric images.

### The $1/f$ Power Law of Natural Images

**Empirical observation:** The singular values of natural images follow an approximate power law:

$$\sigma_i \propto i^{-\alpha}, \quad \alpha \approx 1.0-1.5$$

This is related to the famous $1/f^2$ power spectral density of natural images:
$$S(f) \propto f^{-2}$$

**Derivation:** If the power spectrum falls as $f^{-2}$, the total energy above frequency $f$ is:
$$E(>f) = \int_f^\infty S(f')df' \propto f^{-1}$$

Since the $k$-th singular value captures the energy at the $k$-th "frequency scale," $\sigma_k^2 \propto k^{-1}$, giving $\sigma_k \propto k^{-1/2}$ in the simplest model. More refined analysis accounting for 2D geometry gives $\alpha \approx 1$.

### Implications for Compression

The rapid decay $\sigma_i \propto i^{-\alpha}$ means that the cumulative energy:
$$\frac{\sum_{i=1}^k \sigma_i^2}{\sum_{i=1}^r \sigma_i^2} \approx 1 - \left(\frac{k}{r}\right)^{1-2\alpha}$$

converges quickly. For $\alpha = 1$: capturing 95% of energy requires only $k \approx r/20$ components — explaining why SVD compression is so effective for natural images.

For synthetic images (text, diagrams), the singular values decay even faster ($\alpha > 2$), enabling even higher compression ratios. An RL agent can adapt its compression rank $k$ based on the observed spectral decay rate.

## Tensor Decomposition — Extending SVD to Color Images

For color images $\mathbf{I} \in \mathbb{R}^{m \times n \times 3}$, standard SVD applies only to 2D slices. Tensor decomposition generalizes SVD to multidimensional arrays.

### Step 1: Mode Unfolding

A 3rd-order tensor $\mathcal{X} \in \mathbb{R}^{m \times n \times 3}$ can be "unfolded" into three matrices:

- **Mode-1 unfolding:** $\mathbf{X}_{(1)} \in \mathbb{R}^{m \times 3n}$ — rows correspond to image rows
- **Mode-2 unfolding:** $\mathbf{X}_{(2)} \in \mathbb{R}^{n \times 3m}$ — rows correspond to image columns
- **Mode-3 unfolding:** $\mathbf{X}_{(3)} \in \mathbb{R}^{3 \times mn}$ — rows correspond to color channels

### Step 2: Tucker Decomposition

The Tucker decomposition of a color image:

$$\mathcal{X} \approx \mathcal{G} \times_1 \mathbf{U}_1 \times_2 \mathbf{U}_2 \times_3 \mathbf{U}_3$$

where:
- $\mathcal{G} \in \mathbb{R}^{k_1 \times k_2 \times k_3}$ is the core tensor
- $\mathbf{U}_1 \in \mathbb{R}^{m \times k_1}$, $\mathbf{U}_2 \in \mathbb{R}^{n \times k_2}$, $\mathbf{U}_3 \in \mathbb{R}^{3 \times k_3}$
- $\times_i$ denotes the mode-$i$ tensor-matrix product

### Step 3: Higher-Order SVD (HOSVD)

**Algorithm (De Lathauwer et al., 2000):**
1. Compute SVD of each mode unfolding: $\mathbf{X}_{(i)} = \mathbf{U}_i \boldsymbol{\Sigma}_i \mathbf{V}_i^T$
2. Truncate each $\mathbf{U}_i$ to $k_i$ columns
3. Core tensor: $\mathcal{G} = \mathcal{X} \times_1 \mathbf{U}_1^T \times_2 \mathbf{U}_2^T \times_3 \mathbf{U}_3^T$

**Storage:** $k_1 k_2 k_3 + mk_1 + nk_2 + 3k_3$ values

For an RGB image with $(k_1, k_2, k_3) = (50, 50, 3)$:
$$\text{Storage} = 7500 + 50m + 50n + 9$$

Compared to $3mn$ for the original — significant compression when $m, n \gg 100$.

### Step 4: CP Decomposition Alternative

The CANDECOMP/PARAFAC decomposition is the tensor analogue of matrix rank-1 decomposition:

$$\mathcal{X} \approx \sum_{r=1}^{R} \lambda_r \cdot \mathbf{a}_r \circ \mathbf{b}_r \circ \mathbf{c}_r$$

where $\circ$ is the outer product and $\mathbf{a}_r \in \mathbb{R}^m$, $\mathbf{b}_r \in \mathbb{R}^n$, $\mathbf{c}_r \in \mathbb{R}^3$.

**Storage:** $R(m + n + 3 + 1)$ values — even more compact than Tucker when $R$ is small.

### Step 5: Application in RL

For an RL agent processing color video:
- Apply HOSVD to each frame for compressed state representation
- The color basis $\mathbf{U}_3$ captures the dominant color modes (often luminance + two chrominance)
- This provides a mathematically principled way to combine spatial compression (through $\mathbf{U}_1, \mathbf{U}_2$) with color compression (through $\mathbf{U}_3$)

**Advantage over per-channel SVD:** The tensor approach captures cross-channel correlations, achieving better compression ratios for images where channels are correlated (which is always the case for natural images).

In [ ]:
# Create a sample image for SVD demonstration
size = 128
x = np.linspace(-3, 3, size)
y = np.linspace(-3, 3, size)
X, Y = np.meshgrid(x, y)

# Create image with multiple features
img = (np.sin(X*2)**2 + np.cos(Y*3)**2) * 0.3
img += np.exp(-(X**2 + Y**2)/2) * 0.7  # Gaussian
img = (img / img.max() * 255).astype(np.float64)

# Perform SVD
U, S, Vt = svd(img, full_matrices=False)

# Reconstruct with different number of components
ranks = [1, 2, 5, 10, 20, 50, 80, 128]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for idx, k in enumerate(ranks):
    row, col = idx // 4, idx % 4
    # Reconstruct with k components
    img_k = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]
    
    # Compression ratio
    original_size = size * size
    compressed_size = k * (size + size + 1)
    ratio = compressed_size / original_size * 100
    
    axes[row, col].imshow(img_k, cmap='gray')
    axes[row, col].set_title(f'Rank {k}\n{ratio:.1f}% of original size')
    axes[row, col].axis('off')

plt.suptitle('SVD Image Compression — Fewer Components = Smaller State for RL!',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('svd_compression.png', dpi=150, bbox_inches='tight')
plt.show()

# Plot singular values
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(S, 'b-', linewidth=2)
ax1.set_xlabel('Component index')
ax1.set_ylabel('Singular value σᵢ')
ax1.set_title('Singular Values (Energy per component)')
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)

cumulative_energy = np.cumsum(S**2) / np.sum(S**2) * 100
ax2.plot(cumulative_energy, 'r-', linewidth=2)
ax2.axhline(y=95, color='g', linestyle='--', label='95% energy')
ax2.axhline(y=99, color='b', linestyle='--', label='99% energy')
k_95 = np.argmax(cumulative_energy >= 95)
ax2.axvline(x=k_95, color='g', linestyle=':', alpha=0.5)
ax2.set_xlabel('Number of components (k)')
ax2.set_ylabel('Cumulative energy (%)')
ax2.set_title(f'Only {k_95} components capture 95% of image!')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('svd_energy.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📊 Original image: {size}×{size} = {size*size} values")
print(f"📊 95% energy captured by {k_95} components = {k_95*(size+size+1)} values")
print(f"📊 Compression: {k_95*(size+size+1)/size/size*100:.1f}% of original")

## Rate-Distortion Theory — Information-Theoretic Limits of Image Compression

Rate-distortion theory provides the fundamental limits on how much an image can be compressed while maintaining a given quality level.

### Step 1: The Rate-Distortion Function

For a source $X$ with distortion measure $d(x, \hat{x})$, the rate-distortion function is:

$$R(D) = \min_{p(\hat{x}|x) : E[d(X, \hat{X})] \leq D} I(X; \hat{X})$$

This is the minimum number of bits per symbol needed to represent $X$ with average distortion at most $D$.

### Step 2: Rate-Distortion for Gaussian Source (Image Model)

If pixel values are modeled as i.i.d. Gaussian $X \sim \mathcal{N}(0, \sigma_X^2)$ with MSE distortion:

$$R(D) = \begin{cases} \frac{1}{2}\log_2\frac{\sigma_X^2}{D} & 0 \leq D \leq \sigma_X^2 \\ 0 & D > \sigma_X^2 \end{cases}$$

**Proof sketch:** The optimal test channel is $\hat{X} = X + Z$ where $Z \sim \mathcal{N}(0, D)$ independent of $X$, giving $I(X; \hat{X}) = \frac{1}{2}\log_2(1 + \sigma_X^2/D)$. Lagrangian optimization shows $D = \sigma_Z^2$, yielding the formula above. $\blacksquare$

### Step 3: SVD Compression and Rate-Distortion

For rank-$k$ SVD approximation of an $m \times n$ image:

**Rate (bits per pixel):**
$$R_k = \frac{k(m + n + 1) \cdot b}{m \cdot n} \quad \text{(where } b \text{ = bits per coefficient)}$$

**Distortion (MSE):**
$$D_k = \frac{1}{mn}\sum_{i=k+1}^{r} \sigma_i^2$$

**Operational R-D curve:** Plotting $(R_k, D_k)$ for $k = 1, 2, \ldots, r$ traces the achievable rate-distortion tradeoff for SVD compression.

### Step 4: Comparison with the Shannon Limit

The SVD R-D curve always lies ABOVE the Shannon bound $R(D)$ — meaning SVD compression is suboptimal (but guaranteed to be optimal among linear methods, by Eckart-Young).

The gap between SVD and the Shannon limit represents the potential gain from:
- **Nonlinear compression** (neural codecs, VQ-VAE)
- **Entropy coding** of SVD coefficients
- **Exploiting spatial correlations** beyond what SVD captures

### Step 5: RL for Learned Compression

Modern learned image compression uses RL-like optimization to navigate the R-D tradeoff:

$$\min_{\theta} R(\theta) + \lambda D(\theta) = E[-\log p_{\hat{X}}(\hat{X})] + \lambda E[\|X - \hat{X}\|^2]$$

where $\theta$ parameterizes the encoder-decoder, $R$ is the estimated bitrate, $D$ is the distortion, and $\lambda$ controls the tradeoff. This is equivalent to an RL problem where the "reward" is $-(R + \lambda D)$.

## Randomized SVD and Computational Complexity for Large Images

Computing the full SVD of a large image matrix is expensive. Randomized algorithms make SVD practical for high-resolution images used in RL.

### Step 1: Computational Cost of Full SVD

For an $m \times n$ matrix (with $m \geq n$):
$$T_{\text{full SVD}} = O(mn^2)$$

For a 1080p grayscale image ($1920 \times 1080$): $O(1920 \times 1080^2) \approx 2.2 \times 10^9$ operations — too slow for real-time RL at 30+ fps.

### Step 2: Truncated SVD via Randomized Algorithm

**Halko-Martinsson-Tropp (2011) Algorithm:**

**Input:** Matrix $\mathbf{A} \in \mathbb{R}^{m \times n}$, target rank $k$, oversampling parameter $p$ (typically $p = 5$)

**Stage 1 — Find approximate range:**
1. Generate random matrix $\boldsymbol{\Omega} \in \mathbb{R}^{n \times (k+p)}$ (Gaussian entries)
2. Form $\mathbf{Y} = \mathbf{A}\boldsymbol{\Omega} \in \mathbb{R}^{m \times (k+p)}$ — captures the column space
3. Compute QR: $\mathbf{Y} = \mathbf{Q}\mathbf{R}$ where $\mathbf{Q} \in \mathbb{R}^{m \times (k+p)}$ is orthonormal

**Stage 2 — Form small matrix and decompose:**
4. $\mathbf{B} = \mathbf{Q}^T\mathbf{A} \in \mathbb{R}^{(k+p) \times n}$ — project $\mathbf{A}$ onto range of $\mathbf{Q}$
5. Compute full SVD: $\mathbf{B} = \tilde{\mathbf{U}}\boldsymbol{\Sigma}\mathbf{V}^T$
6. Set $\mathbf{U} = \mathbf{Q}\tilde{\mathbf{U}}$

**Output:** $\mathbf{A} \approx \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^T$ (rank-$k$ approximation)

### Step 3: Complexity Analysis

$$T_{\text{randomized}} = O(mn(k+p)) + O((k+p)^2 n)$$

For 1080p with $k = 50, p = 5$: $O(1920 \times 1080 \times 55) \approx 1.1 \times 10^8$ — a **20× speedup** over full SVD.

### Step 4: Error Bound

**Theorem:** With probability $\geq 1 - 3p^{-p}$:

$$E\|\mathbf{A} - \mathbf{Q}\mathbf{Q}^T\mathbf{A}\|_F \leq \left(1 + \frac{k}{p-1}\right)^{1/2} \left(\sum_{j=k+1}^{\min(m,n)} \sigma_j^2\right)^{1/2}$$

The factor $(1 + k/(p-1))^{1/2}$ is close to 1 for $p \geq 5$, meaning the randomized algorithm is nearly as good as the optimal truncated SVD. $\blacksquare$

### Step 5: RL Pipeline Application

In a typical RL vision pipeline:
1. **Observation:** Camera captures $1920 \times 1080$ frame
2. **Compression:** Randomized rank-50 SVD in ~10ms
3. **State vector:** $\mathbf{z} = [\sigma_1, \ldots, \sigma_{50}, \mathbf{U}_{:,1:50}^T\mathbf{x}] \in \mathbb{R}^{50}$
4. **Policy:** $\pi(\mathbf{z}) \to a$ — fast decision on compressed state

The randomized SVD makes this pipeline feasible at real-time rates, enabling SVD-based state compression in practical RL systems.

## SVD Computational Complexity and Randomized Methods

For large images, the computational cost of SVD matters. Understanding complexity helps RL practitioners make informed decisions about state compression.

### Step 1: Classical SVD Complexity

The standard SVD algorithm (Golub-Kahan bidiagonalization + QR iteration) has complexity:

$$T_{\text{SVD}}(m, n) = O(\min(m^2 n, mn^2))$$

For a $512 \times 512$ image: $T \approx O(512^3) \approx 1.3 \times 10^8$ operations.

For a full $1920 \times 1080$ HD frame: $T \approx O(1920 \times 1080^2) \approx 2.2 \times 10^9$ — too slow for real-time RL.

### Step 2: Truncated SVD — When We Only Need Top-k

For rank-$k$ approximation, we only need $\sigma_1, \ldots, \sigma_k$ and corresponding $\mathbf{u}_i, \mathbf{v}_i$:

**Lanczos/Arnoldi iteration:** Computes top-$k$ singular triplets in $O(mnk)$ time.

**Complexity comparison:**

| Method | Time | Space | Best For |
|:-------|:-----|:------|:---------|
| Full SVD | $O(mn \cdot \min(m,n))$ | $O(mn)$ | Complete decomposition |
| Truncated (Lanczos) | $O(mnk)$ | $O((m+n)k)$ | Known small $k$ |
| Randomized SVD | $O(mn\log k)$ | $O((m+n)k)$ | Approximate, very fast |

### Step 3: Randomized SVD Algorithm (Halko, Martinsson, Tropp 2011)

**Algorithm:**
1. Generate random Gaussian matrix $\boldsymbol{\Omega} \in \mathbb{R}^{n \times (k+p)}$ (where $p \sim 5-10$ is oversampling)
2. Form $\mathbf{Y} = \mathbf{A}\boldsymbol{\Omega}$ — projects $\mathbf{A}$ onto a random subspace [$O(mn(k+p))$]
3. Compute QR: $\mathbf{Y} = \mathbf{Q}\mathbf{R}$ [$O(m(k+p)^2)$]
4. Form $\mathbf{B} = \mathbf{Q}^T\mathbf{A}$ [$O(mn(k+p))$]
5. Compute SVD of $\mathbf{B}$ (small: $(k+p) \times n$): $\mathbf{B} = \hat{\mathbf{U}}\boldsymbol{\Sigma}\mathbf{V}^T$
6. Recover $\mathbf{U} = \mathbf{Q}\hat{\mathbf{U}}$

**Error bound:**
$$E[\|\mathbf{A} - \mathbf{Q}\mathbf{Q}^T\mathbf{A}\|_F] \leq \left(1 + \frac{k}{p-1}\right)^{1/2} \left(\sum_{i=k+1}^{\min(m,n)} \sigma_i^2\right)^{1/2}$$

With oversampling $p = 5$: the error is at most $\sqrt{1 + k/4}$ times the optimal rank-$k$ error — typically within a few percent of optimal. $\blacksquare$

### Step 4: RL Implications

For real-time RL with image states:
- **Frame rate requirement:** 30+ FPS → budget $\leq 33$ms per frame
- **Randomized SVD** with $k = 20$, $p = 5$ on a $84 \times 84$ grayscale Atari frame: $\approx 0.5$ms — feasible for real-time compression
- The compressed state $\mathbf{z} = \boldsymbol{\Sigma}_k \mathbf{V}_k^T \in \mathbb{R}^{k \times n}$ is a compact, mathematically principled state representation

This is the linear precursor to the learned compression in modern deep RL (autoencoders, VAEs), where nonlinear SVD analogs capture more complex structure.

## Summary

### Key equations:
- Image as matrix: $\mathbf{I} \in \mathbb{R}^{M \times N}$
- SVD: $\mathbf{I} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^T$
- Compression: Use top-$k$ singular values
- **For RL**: Compressed representation = efficient state space

---
**Next:** Module 1.5 — Image Histograms